# 02 Currency dimension


In [ ]:
%pip install -q google-cloud-bigquery pandas pyarrow db-dtypes requests papermill
import os,sys,json,pandas as pd
from google.cloud import bigquery
PROJECT_ID="your-gcp-project-id"  # замініть у всіх ноутбуках
LOCATION="EU"; creds=None
if "google.colab" in sys.modules:
 from google.colab import auth; auth.authenticate_user()
elif os.environ.get("GCP_SA_KEY"):
 from google.oauth2 import service_account
 creds=service_account.Credentials.from_service_account_info(json.loads(os.environ["GCP_SA_KEY"]),scopes=["https://www.googleapis.com/auth/cloud-platform"])
client=bigquery.Client(project=PROJECT_ID,location=LOCATION,credentials=creds)
print("проєкт:",client.project)

In [ ]:
R=f"{PROJECT_ID}.nbu_raw.raw_rates"; T=f"{PROJECT_ID}.nbu_dwh.dim_currency"
r=client.query(f"SELECT ingested_at,business_date,payload FROM `{R}`").to_dataframe(); assert not r.empty
p=pd.json_normalize(r.payload.map(json.loads)); d=pd.concat([r[["ingested_at"]].reset_index(drop=True),p],axis=1)
d["currency_code"]=d.cc.astype(str).str.strip().str.upper(); d["currency_name"]=d.txt.astype(str).str.strip(); d["r030"]=pd.to_numeric(d.r030).astype("Int64")
o=d.sort_values("ingested_at").drop_duplicates("currency_code",keep="last")[["currency_code","currency_name","r030"]].sort_values("currency_code").reset_index(drop=True); o.insert(0,"currency_key",range(1,len(o)+1)); o=pd.concat([pd.DataFrame([[-1,"N/A","Unknown",-1]],columns=o.columns),o],ignore_index=True)
s=[bigquery.SchemaField("currency_key","INTEGER"),bigquery.SchemaField("currency_code","STRING"),bigquery.SchemaField("currency_name","STRING"),bigquery.SchemaField("r030","INTEGER")]; client.load_table_from_dataframe(o,T,job_config=bigquery.LoadJobConfig(schema=s,write_disposition="WRITE_TRUNCATE")).result()
assert o.currency_code.is_unique and (o.currency_key==-1).any() and len(o)==d.currency_code.nunique()+1; display(o.head()); print("checks: OK")